# 6. Feature Engineering

In [13]:
import pandas as pd

riders = pd.read_csv("../data/processed/riders_clean.csv")
trips = pd.read_csv("../data/processed/trips_clean.csv")
sessions = pd.read_csv("../data/processed/sessions_clean.csv")
promotions = pd.read_csv("../data/processed/promotions_clean.csv")
last_trip = pd.read_csv("../data/processed/churn_base.csv")

# Convert dates again
riders['signup_date'] = pd.to_datetime(riders['signup_date'], errors='coerce')
trips['pickup_time'] = pd.to_datetime(trips['pickup_time'], errors='coerce')
sessions['session_time'] = pd.to_datetime(sessions['session_time'], errors='coerce')
last_trip['pickup_time'] = pd.to_datetime(last_trip['pickup_time'], errors='coerce')

C:\Users\HP\AppData\Local\Temp\ipykernel_21708\3530920889.py:12: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  sessions['session_time'] = pd.to_datetime(sessions['session_time'], errors='coerce')


In [15]:
trip_features = (trips.groupby('user_id')
                 .agg(
                 total_trips = ('trip_id', 'count'),
                 avg_fare = ('fare', 'mean'),
                 total_spend = ('fare', 'sum'),
                 avg_surge = ('surge_multiplier', 'mean'),
                 avg_tip = ('tip', 'mean'),
                 cities_used = ('city', 'nunique')
                )
                 .reset_index()
                )
trip_features.head()

,user_id,total_trips,avg_fare,total_spend,avg_surge,avg_tip,cities_used
0,R00000,25,14.642000,366.05,1.096000,0.161200,1
1,R00001,14,12.895000,180.53,1.071429,0.054286,1
2,R00002,24,15.791250,378.99,1.191667,0.217083,1
3,R00003,9,13.496667,121.47,1.155556,0.096667,1
4,R00004,16,16.776875,268.43,1.262500,0.586250,1


In [16]:
session_features = (sessions.groupby('user_id')
                   .agg(
                   total_sessions = ('session_id', 'count'),
                   avg_time_on_app = ('time_on_app', 'mean'),
                   avg_pages_visited = ('pages_visited', 'mean'),
                    conversion_rate = ('converted', 'mean')
                   )
                    .reset_index()
                   )
session_features.head()

,user_id,total_sessions,avg_time_on_app,avg_pages_visited,conversion_rate
0,R00000,4,92.000000,3.000000,0.25
1,R00001,3,174.666667,2.666667,0.00
2,R00002,3,191.000000,3.000000,0.00
3,R00003,3,75.333333,1.666667,0.00
4,R00004,2,17.000000,2.500000,0.00


In [17]:
# Ensured IDs are consistent in all tables used for merges
last_trip['user_id'] = last_trip['user_id'].astype(str).str.strip().str.upper()
trip_features['user_id'] = trip_features['user_id'].astype(str).str.strip().str.upper()
session_features['user_id'] = session_features['user_id'].astype(str).str.strip().str.upper()
riders['user_id'] = riders['user_id'].astype(str).str.strip().str.upper()

# churn base + dates
model_df = last_trip[['user_id', 'pickup_time', 'days_since_last_trip', 'churn']].merge(
    riders[['user_id', 'signup_date']],
    on='user_id',
    how='left'
)

# Add engineered features
model_df = model_df.merge(trip_features, on='user_id', how='left')
model_df = model_df.merge(session_features, on='user_id', how='left')

print(model_df.columns)

Index(['user_id', 'pickup_time', 'days_since_last_trip', 'churn',
       'signup_date', 'total_trips', 'avg_fare', 'total_spend', 'avg_surge',
       'avg_tip', 'cities_used', 'total_sessions', 'avg_time_on_app',
       'avg_pages_visited', 'conversion_rate'],
      dtype='object')


In [18]:
model_df.to_csv("../data/processed/model_df.csv", index=False)
print("Saved model_df.csv")

Saved model_df.csv
